# Simpsons Character Classification

This project classifies characters from *The Simpsons* using transfer learning with PyTorch. The goal is to build a practical image-classification pipeline: dataset loading, stratified validation, class-imbalance handling, model fine-tuning, error analysis, and Kaggle-style submission generation.

The repository is structured as a small portfolio project: the notebook explains the experiment end to end, while reusable training, visualization, and label-audit code lives in separate Python modules.


## What this demonstrates

- Transfer learning with a pretrained computer-vision model.
- Clean train/validation split with stratification.
- Handling class imbalance with weighted sampling and weighted loss.
- Reusable project modules instead of notebook-only helper code.
- Error analysis workflow for identifying suspicious labels.
- Reproducible Colab setup with GitHub-hosted code and dataset release assets.


The public test set contains 991 images. The model predicts one character label per image and writes the final predictions to a submission CSV.


## 1. Environment Setup


This section imports the core libraries, verifies GPU availability, and mounts Google Drive for persistent artifacts such as checkpoints, audit tables, and the cached dataset archive.


In [ ]:
# we will verify that GPU is enabled for this notebook
# following should print: CUDA is available!  Training on GPU ...
#
# if it prints otherwise, then you need to enable GPU:
# from Menu > Runtime > Change Runtime Type > Hardware Accelerator > GPU

import torch
import numpy as np

train_on_gpu = torch.cuda.is_available()

if not train_on_gpu:
    print('CUDA is not available.  Training on CPU ...')
else:
    print('CUDA is available!  Training on GPU ...')


from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!nvidia-smi


In [ ]:
import numpy as np
import pandas as pd
import torch.nn as nn
from PIL import Image
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision.transforms import v2
from matplotlib import pyplot as plt
%matplotlib inline


## 2. Project Configuration

The notebook loads source code from GitHub and stores only large runtime artifacts on Google Drive. This keeps the repository lightweight while making Colab runs reproducible.


In [ ]:
# Dataset modes
DATA_MODES = ["train", "val", "test"]

# Training device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Dataset paths after extracting the archive in Colab
TRAIN_DIR = Path("/content/train")
TEST_DIR = Path("/content/testset")

# Load project code from GitHub. Use Drive only for large runtime artifacts.
import subprocess
import sys

REPO_URL = "https://github.com/eritry/simpsons-classification.git"
CODE_DIR = Path("/content/simpsons-classification")

if not (CODE_DIR / "utils.py").exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(CODE_DIR)],
        check=True,
    )

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

# Single Google Drive root for persistent artifacts.
DRIVE_ARTIFACT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/simpsons")
CHECKPOINT_DIR = DRIVE_ARTIFACT_DIR / "checkpoints"
LABEL_AUDIT_DIR = DRIVE_ARTIFACT_DIR / "label_audit"
DATASET_DIR = DRIVE_ARTIFACT_DIR / "dataset"
DATASET_RELEASE_URL = "https://github.com/eritry/simpsons-classification/releases/download/dataset-v1/journey-springfield.zip"
DATASET_ARCHIVE_PATH = DATASET_DIR / "journey-springfield.zip"

EFFICIENTNET_STAGE1_DIR = CHECKPOINT_DIR / "efficientnet_v2_s_stage1"
EFFICIENTNET_STAGE2_DIR = CHECKPOINT_DIR / "efficientnet_v2_s_stage2"
BEST_MODEL_PATH = EFFICIENTNET_STAGE2_DIR / "best_model.pth"

for directory in [DRIVE_ARTIFACT_DIR, CHECKPOINT_DIR, LABEL_AUDIT_DIR, DATASET_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# CSV for manually reviewed suspicious labels. Created later in the audit section.
REVIEW_PATH = LABEL_AUDIT_DIR / "suspicious_manual.csv"

# ImageNet normalization parameters
NORMALIZE_MEAN = [0.485, 0.456, 0.406]
NORMALIZE_STD = [0.229, 0.224, 0.225]

# Model input size
RESCALE_SIZE = [224, 224]


## 3. Data Loading and Preprocessing


### Dataset Download and Cache

The dataset archive is hosted as a GitHub Release asset. On the first run, the notebook downloads it to Google Drive; later runs reuse the cached archive and only extract it into the current Colab runtime.


In [ ]:
import urllib.request

if not DATASET_ARCHIVE_PATH.exists():
    print("Downloading dataset from GitHub Release:", DATASET_RELEASE_URL)
    urllib.request.urlretrieve(DATASET_RELEASE_URL, DATASET_ARCHIVE_PATH)
else:
    print("Dataset archive already cached on Drive:", DATASET_ARCHIVE_PATH)

print("Dataset archive size, MB:", round(DATASET_ARCHIVE_PATH.stat().st_size / 1024 / 1024, 1))


In [ ]:
import zipfile

sample_submission_path = Path("/content/sample_submission.csv")

if not TRAIN_DIR.exists() or not TEST_DIR.exists() or not sample_submission_path.exists():
    print("Extracting dataset to /content ...")
    with zipfile.ZipFile(DATASET_ARCHIVE_PATH, "r") as archive:
        archive.extractall("/content")
else:
    print("Dataset already extracted in this runtime.")


In [ ]:
# Optional label-fix block based on a manually reviewed CSV.
# By default no files are moved: fill REVIEW_PATH and set APPLY_LABEL_FIXES=True first.
import shutil

APPLY_LABEL_FIXES = False


def move_reviewed_files_to_class(review_df, dry_run=True):
    """
    Moves only rows with action == "MOVE".
    The target class comes from correct_label when available, otherwise from predicted_label.
    """
    required_columns = ["path", "predicted_label", "action"]
    missing_columns = [col for col in required_columns if col not in review_df.columns]
    if missing_columns:
        raise ValueError(f"review_df is missing required columns: {missing_columns}")

    actions = review_df["action"].fillna("").astype(str).str.upper()
    rows_to_move = review_df[actions == "MOVE"].copy()
    print("Files to move:", len(rows_to_move))

    moved_paths = {}

    for _, row in rows_to_move.iterrows():
        src_path = Path(row["path"])
        correct_label = row.get("correct_label", "")
        target_class = "" if pd.isna(correct_label) else str(correct_label).strip()
        if not target_class:
            target_class = str(row["predicted_label"]).strip()

        if not src_path.exists():
            print("File not found, skipping:", src_path)
            continue

        current_class_dir = src_path.parent
        split_root_dir = current_class_dir.parent
        dst_dir = split_root_dir / target_class
        dst_path = dst_dir / src_path.name

        if not dst_dir.exists():
            print("Target class directory not found, skipping:", dst_dir)
            continue

        if current_class_dir.name == target_class:
            print("File is already in the target directory, skipping:", src_path)
            continue

        if dst_path.exists():
            stem = src_path.stem
            suffix = src_path.suffix
            counter = 1
            while dst_path.exists():
                dst_path = dst_dir / f"{stem}_moved_{counter}{suffix}"
                counter += 1

        print("FROM:", src_path)
        print("TO:  ", dst_path)

        if not dry_run:
            shutil.move(str(src_path), str(dst_path))
            moved_paths[src_path] = dst_path

    if dry_run:
        print("This was dry_run=True. No files were moved.")
    else:
        print("Done. Files were moved.")

    return moved_paths


In [ ]:
if APPLY_LABEL_FIXES:
    if not REVIEW_PATH.exists():
        raise FileNotFoundError(f"Manual review file not found: {REVIEW_PATH}")

    review_df = pd.read_csv(REVIEW_PATH)
    moved_paths = move_reviewed_files_to_class(review_df=review_df, dry_run=False)
else:
    moved_paths = {}
    print("Label fixing is disabled. Set APPLY_LABEL_FIXES = True to enable it.")


In [ ]:
train_val_files = sorted(list(TRAIN_DIR.rglob('*.jpg')))
test_files = sorted(list(TEST_DIR.rglob('*.jpg')))
print("Total training/validation images:", len(train_val_files))
print("Test images:", len(test_files))


### Label Encoding

Class names are inferred from parent directory names. `LabelEncoder` maps these text labels to integer class IDs for training and later converts model predictions back to class names for the submission file.


In [ ]:
label_encoder = LabelEncoder()

train_val_labels = [path.parent.name for path in train_val_files]

label_encoder.fit(train_val_labels)


### Train/Validation Split

The original training folder is split into training and validation subsets. The split is stratified by class label so that rare and frequent characters are represented proportionally in both subsets.


In [ ]:
from sklearn.model_selection import train_test_split

train_files, val_files = train_test_split(
    train_val_files,
    test_size=0.25,
    stratify=train_val_labels,
    random_state=42,
)


### Dataset and DataLoader Objects

`SimpsonsDataset` applies stronger image augmentation during training and deterministic preprocessing for validation/test inference. This mirrors a standard production pattern: augment only the training signal, keep evaluation stable.


In [ ]:
class SimpsonsDataset(Dataset):
    def __init__(self, files, label_encoder, mode):
        super().__init__()

        self.files = sorted(files)
        self.mode = mode

        if self.mode not in DATA_MODES:
            print(f"{self.mode} is not correct; correct modes: {DATA_MODES}")
            raise NameError

        self.label_encoder = label_encoder
        self.len_ = len(self.files)

        self.train_transform = v2.Compose([
            v2.Resize([256, 256]),
            v2.RandomResizedCrop(RESCALE_SIZE, scale=(0.75, 1.0), ratio=(0.9, 1.1)),
            v2.RandomHorizontalFlip(p=0.5),
            v2.RandomRotation(degrees=10),
            v2.ColorJitter(
                brightness=0.2,
                contrast=0.2,
                saturation=0.2,
                hue=0.03,
            ),
            v2.PILToTensor(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(NORMALIZE_MEAN, NORMALIZE_STD),
        ])

        self.val_transform = v2.Compose([
            v2.Resize(RESCALE_SIZE),
            v2.PILToTensor(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(NORMALIZE_MEAN, NORMALIZE_STD),
        ])

    def __len__(self):
        return self.len_

    def __getitem__(self, index):
        x = self.load_image(self.files[index])
        x = self.transform_images_to_tensors(x)

        if self.mode == "test":
            return x

        path = self.files[index]
        y = self.label_encoder.transform([path.parent.name]).item()
        return x, y

    def load_image(self, file):
        image = Image.open(file).convert("RGB")
        image.load()
        return image

    def transform_images_to_tensors(self, image):
        if self.mode == "train":
            return self.train_transform(image)
        return self.val_transform(image)


In [ ]:
train_dataset = SimpsonsDataset(train_files, label_encoder = label_encoder, mode='train')
val_dataset = SimpsonsDataset(val_files, label_encoder, mode='val')


In [ ]:
batch_size = 64


### Class Distribution

The dataset is imbalanced: major characters have many more samples than rare characters. The plot below makes that imbalance explicit before training.


In [ ]:
from collections import Counter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


def get_simpsons_counts(dataset):
    class_names = list(dataset.label_encoder.classes_)

    class_names_from_paths = [
        path.parent.name
        for path in dataset.files
    ]

    counts_by_class_name = Counter(class_names_from_paths)

    counts = [
        counts_by_class_name[class_name]
        for class_name in class_names
    ]

    return class_names, counts


def plot_train_val_distribution(train_dataset, val_dataset, title="Train vs Validation class distribution"):
    train_class_names, train_counts = get_simpsons_counts(train_dataset)
    val_class_names, val_counts = get_simpsons_counts(val_dataset)

    assert train_class_names == val_class_names, "Train and validation classes do not match"

    df = pd.DataFrame({
        "class_name": train_class_names,
        "train_count": train_counts,
        "val_count": val_counts,
    })

    df["total_count"] = df["train_count"] + df["val_count"]
    df = df.sort_values("total_count", ascending=False).reset_index(drop=True)

    x = np.arange(len(df))
    width = 0.42

    plt.figure(figsize=(18, 6))
    plt.bar(x - width / 2, df["train_count"], width, label="train")
    plt.bar(x + width / 2, df["val_count"], width, label="val")

    plt.xticks(x, df["class_name"], rotation=90)
    plt.xlabel("Character")
    plt.ylabel("Number of images")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()

    # display(df[["class_name", "train_count", "val_count", "total_count"]])

    print("Train total:", int(df["train_count"].sum()))
    print("Validation total:", int(df["val_count"].sum()))
    print("Total:", int(df["total_count"].sum()))

    print("\nTrain min/max:", int(df["train_count"].min()), "/", int(df["train_count"].max()))
    print("Val min/max:", int(df["val_count"].min()), "/", int(df["val_count"].max()))

    return df


distribution_df = plot_train_val_distribution(
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    title="Train vs Validation distribution by Simpsons character"
)


### Imbalance Handling

A `WeightedRandomSampler` increases the probability of sampling under-represented classes. I use inverse square-root class frequency rather than full inverse frequency to reduce imbalance without making rare classes dominate every batch.


In [ ]:
train_numeric_labels = label_encoder.transform([
    path.parent.name for path in train_dataset.files
])

class_counts = np.bincount(
    train_numeric_labels,
    minlength=len(label_encoder.classes_)
)

class_weights_for_sampler = 1.0 / np.sqrt(np.maximum(class_counts, 1))

sample_weights = torch.DoubleTensor([
    class_weights_for_sampler[label]
    for label in train_numeric_labels
])

train_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    sampler=train_sampler,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)


val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

loaders = {
    "train": train_loader,
    "val": val_loader,
}

print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))
print("Number of classes:", len(label_encoder.classes_))


### Visual Sanity Check

Before training, it is useful to inspect augmented batches manually. This catches obvious preprocessing issues such as wrong labels, broken images, or overly aggressive transforms.


Visualization helpers are implemented in `visualization.py`. Keeping them outside the notebook makes the notebook easier to read and the helpers easier to reuse.


In [ ]:
from visualization import (
    show_images_from_loader,
    show_images_with_predictions,
)


In [ ]:
show_images_from_loader(n_rows=3, n_cols=4, loader=train_loader, label_encoder=label_encoder)


## 4. Model Architecture


### EfficientNetV2-S Transfer Learning

The model uses ImageNet-pretrained EfficientNetV2-S. The classifier head is replaced with a task-specific linear layer for the Simpsons character classes.


### Why this model

EfficientNetV2-S is a strong compact convolutional architecture with pretrained ImageNet features. For a medium-sized character dataset, transfer learning usually gives better sample efficiency than training a CNN from scratch.


In [ ]:
# Use EfficientNet instead of the original simple CNN baseline
# Training, evaluation, prediction, plotting, and checkpointing helpers are imported from utils.py.


In [ ]:
from torchvision import models
from torchvision.models import EfficientNet_V2_S_Weights
import torch.nn as nn

def create_model(n_classes):
    weights = EfficientNet_V2_S_Weights.IMAGENET1K_V1
    model = models.efficientnet_v2_s(weights=weights)

    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, n_classes)
    )

    return model

model_simple_cnn = create_model(len(label_encoder.classes_)).to(DEVICE)


## 5. Training Utilities

Training, evaluation, metric logging, checkpointing, plotting, and prediction helpers are imported from `utils.py`. The notebook keeps experiment orchestration visible while moving reusable mechanics into a module.


In [ ]:
from utils import (
    make_training_history,
    plot_history,
    predict,
    train_CNN,
)


In [ ]:
history = make_training_history()


## 6. Training and Validation


### Optimization Strategy

Training uses two stages. First, only the new classifier head is trained. Then the full EfficientNet backbone is unfrozen and fine-tuned with a lower learning rate for pretrained layers and a higher learning rate for the classifier.


### Validation metric

The training loop tracks both accuracy and macro F1. Macro F1 is important here because the dataset is imbalanced: it gives rare classes the same weight as frequent classes when summarizing performance.


In [ ]:
# Use weighted CrossEntropyLoss for the fine-tuning stage

class_weights = 1.0 / np.sqrt(np.maximum(class_counts, 1))
class_weights = class_weights / class_weights.mean()

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=DEVICE
)

# The scheduler must be created after the optimizer. Use a factory and call it
# after creating the optimizer for a specific training stage.
def make_cosine_scheduler(optimizer, t_max, eta_min=1e-6):
    return torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=t_max,
        eta_min=eta_min,
    )


In [ ]:
# Stage 1: train only the new EfficientNet classifier head.

for param in model_simple_cnn.parameters():
    param.requires_grad = False

for param in model_simple_cnn.classifier.parameters():
    param.requires_grad = True

criterion = nn.CrossEntropyLoss(label_smoothing=0.02)

optimizer = torch.optim.AdamW(
    model_simple_cnn.classifier.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

history = train_CNN(
    model=model_simple_cnn,
    num_epochs=3,
    dataloaders=loaders,
    optimizer=optimizer,
    loss_func=criterion,
    device=DEVICE,
    val_every_steps=100,
    save_dir=EFFICIENTNET_STAGE1_DIR,
)

# Stage 2: unfreeze EfficientNet and fine-tune with lower learning rates.

for param in model_simple_cnn.parameters():
    param.requires_grad = True

optimizer = torch.optim.AdamW(
    [
        {"params": model_simple_cnn.features.parameters(), "lr": 1e-5},
        {"params": model_simple_cnn.classifier.parameters(), "lr": 3e-4},
    ],
    weight_decay=1e-4
)

criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=0.02
)

stage2_scheduler = make_cosine_scheduler(optimizer, t_max=10)

history = train_CNN(
    model=model_simple_cnn,
    num_epochs=10,
    dataloaders=loaders,
    optimizer=optimizer,
    loss_func=criterion,
    device=DEVICE,
    val_every_steps=100,
    save_dir=EFFICIENTNET_STAGE2_DIR,
    history=history,
    scheduler=stage2_scheduler,
)

plot_history(history)


In [ ]:
from sklearn.metrics import classification_report

model = model_simple_cnn
model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for images, targets in val_loader:
        images = images.to(DEVICE)
        targets = targets.to(DEVICE)

        logits = model(images)
        preds = logits.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(targets.cpu().numpy())

print(classification_report(
    all_targets,
    all_preds,
    target_names=label_encoder.classes_,
    digits=4
))


### Reference Performance

A representative run of this pipeline reached approximately 0.95 validation accuracy and 0.85 macro F1. The corresponding Kaggle submission score recorded during the project was 0.97236.

These metrics are included as a reference point; exact numbers may vary slightly with runtime, random seed, and future dataset-cleaning decisions.


## 7. Error Analysis and Label Audit

After training, the model is used to find examples where it disagrees with the current folder label while being highly confident. These cases are good candidates for manual label review. This is not automatic relabeling: the CSV is meant to support human inspection.


In [ ]:
from label_audit import (
    run_label_audit,
    show_review_examples,
)


In [ ]:
review_csv_path = LABEL_AUDIT_DIR / "suspicious_manual_review.csv"
print("Label audit directory:", LABEL_AUDIT_DIR)


In [ ]:
audit_dataset, df_all, df_suspicious = run_label_audit(
    model=model_simple_cnn,
    train_files=train_val_files,
    label_encoder=label_encoder,
    dataset_cls=SimpsonsDataset,
    device=DEVICE,
    batch_size=64,
    num_workers=0,
    min_confidence=0.90,
    min_margin=0.20,
    top_n=300,
    save_all_csv_path=LABEL_AUDIT_DIR / "all_predictions.csv",
    save_review_csv_path=LABEL_AUDIT_DIR / "suspicious_labels_review.csv",
)


Label-audit helpers are implemented in `label_audit.py`. They scan predictions, rank suspicious examples, save review CSV files, and visualize candidates for manual inspection.


In [ ]:
review_df = df_suspicious.copy()
review_df["action"] = ""          # options: "move", "keep", "skip"
review_df["correct_label"] = ""   # for example: "homer_simpson"

review_csv_path = LABEL_AUDIT_DIR / "suspicious_manual_new.csv"
review_df.to_csv(review_csv_path, index=False)

show_review_examples(
    review_df=review_df,
    dataset=audit_dataset,
    start_pos=0,
    n_rows=4,
    n_cols=3,
)


### Prediction Confidence Visualization

The next visualization samples validation images and overlays the model's top prediction and confidence. This is a quick qualitative check of calibration and common confusions.


`show_images_with_predictions` is implemented in `visualization.py`.


In [ ]:
show_images_with_predictions(
    n_rows=3,
    n_cols=3,
    dataset=val_dataset,
    model=model_simple_cnn,
    label_encoder=label_encoder,
    device=DEVICE,
)


## 8. Kaggle-Style Submission


Create a deterministic test DataLoader. Test samples have no labels, so the dataset returns only transformed images.


In [ ]:
test_dataset = SimpsonsDataset(test_files, label_encoder = label_encoder, mode="test")
test_loader = DataLoader(test_dataset, shuffle=False, batch_size=64)


The `predict` helper returns integer class IDs for all test images.


Load the best checkpoint if it exists, then run inference on the test set.


In [ ]:
if BEST_MODEL_PATH.exists():
    model_simple_cnn.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))
    model_simple_cnn.to(DEVICE)
    model_simple_cnn.eval()
else:
    print(f"Best model checkpoint not found: {BEST_MODEL_PATH}. Using current model state.")

predicted_numeric_labels = predict(model_simple_cnn, test_loader, device=DEVICE)
predicted_numeric_labels = predicted_numeric_labels.numpy()


Convert numeric predictions back to text labels.


In [ ]:
predicted_text_labels = label_encoder.inverse_transform(predicted_numeric_labels)


Load the sample submission format and create a final CSV with the required `Id` and `Expected` columns.


In [ ]:
import pandas as pd
sample_submission = pd.read_csv("/content/sample_submission.csv")
sample_submission.head(10)


In [ ]:
my_submission = pd.DataFrame({'Id': [path.name for path in test_files], 'Expected': predicted_text_labels})
my_submission.head(10)


In [ ]:
# The CSV below can be uploaded to Kaggle or used for local review.


In [ ]:
my_submission.to_csv('simple_cnn_baseline.csv', index=False)
